In [39]:
import pandas as pd 
import numpy as np

### Step 1:Load & Clean

In [40]:
df=pd.read_csv("adult_with_headers (1).csv")
print(df.head())
print (df.info())
print(df.describe())


   age          workclass  fnlwgt   education  education_num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital_status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital_gain  capital_loss  hours_per_week  native_country  income  
0          2174             0              40   United-States   <=50

In [41]:
df.columns = df.columns.str.strip()
for col in df.select_dtypes('object').columns:
    df[col] = df[col].str.strip()
df.replace('?', np.nan, inplace=True)

In [42]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64

## Key Observations from Exploration:
### Dataset has 32,561 records and 15 columns
### Missing values are encoded as '?' in the original file — replaced with NaN
### Numerical features: age, fnlwgt, education_num, capital_gain, capital_loss, hours_per_week
### Categorical features: 8 columns (workclass, education, marital_status, etc.)
### Target is binary: <=50K (75.9%) vs >50K (24.1%) — class imbalance present



In [43]:
df.skew(numeric_only=True).sort_values(ascending=False)

capital_gain      11.953848
capital_loss       4.594629
fnlwgt             1.446980
age                0.558743
hours_per_week     0.227643
education_num     -0.311676
dtype: float64

### STEP 2: Handle Missing Values 

In [44]:
# mode imputation for categorical columns
df['workclass'].fillna(df['workclass'].mode()[0], inplace=True)
df['occupation'].fillna(df['occupation'].mode()[0], inplace=True)
df['native_country'].fillna(df['native_country'].mode()[0], inplace=True)
# After imputation 0 missing values accross all the columns

C:\Users\Mitali Patil\AppData\Local\Temp\ipykernel_11212\2011709343.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['workclass'].fillna(df['workclass'].mode()[0], inplace=True)
C:\Users\Mitali Patil\AppData\Local\Temp\ipykernel_11212\2011709343.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alway

### STEP 3: Feature Engineering 

In [45]:
df['capital_net'] = df['capital_gain'] - df['capital_loss']
df['work_intensity'] = df['age'] * df['hours_per_week']
df['capital_gain_log'] = np.log1p(df['capital_gain'])


### capital_net=Real financial value
### work_intensity=Combined effort
### capital_gain_log=Reduce skewness

### STEP 4: Scaling 

In [46]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
nums = ['age','fnlwgt','education_num',
        'capital_gain','capital_loss','hours_per_week']
ss = StandardScaler()
mm = MinMaxScaler()
df_ss = df.copy(); df_ss[nums] = ss.fit_transform(df[nums])
df_mm = df.copy(); df_mm[nums] = mm.fit_transform(df[nums])


In [47]:
df_ss.describe()
df_mm.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,capital_net,work_intensity,capital_gain_log
count,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000
mean,0.295639,0.120545,0.605379,0.010777,0.020042,0.402423,990.345014,1571.723411,0.734621
std,0.186855,0.071685,0.171515,0.073854,0.092507,0.125994,7408.986951,740.732449,2.454738
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-4356.000000,21.000000,0.000000
25%,0.150685,0.071679,0.533333,0.000000,0.000000,0.397959,0.000000,1050.000000,0.000000
50%,0.273973,0.112788,0.600000,0.000000,0.000000,0.397959,0.000000,1520.000000,0.000000
75%,0.424658,0.152651,0.733333,0.000000,0.000000,0.448980,0.000000,2016.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,99999.000000,8910.000000,11.512925


### STEP 5: Encoding 

In [48]:
df = pd.get_dummies(df, columns=['sex'], prefix='sex')
le = LabelEncoder()
for col in ['workclass','education','marital_status',
            'occupation','relationship','race','native_country']:
    df[col] = le.fit_transform(df[col].astype(str))
df['income'] = (df['income'] == '>50K').astype(int)


### STEP 6: Isolation Forest 

In [49]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
df['outlier'] = iso.fit_predict(df[nums])
df_clean = df[df['outlier'] == 1].drop('outlier', axis=1)
print(f'Clean dataset shape: {df_clean.shape}')


Clean dataset shape: (30933, 19)


### STEP 7: PPS Analysis 

In [50]:
!pip install ppscore

Defaulting to user installation because normal site-packages is not writeable


In [51]:
df.corr(numeric_only=True)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,capital_gain,capital_loss,hours_per_week,native_country,income,capital_net,work_intensity,capital_gain_log,sex_Female,sex_Male,outlier
age,1.000000,0.040504,-0.076646,-0.010508,0.036527,-0.266288,0.001739,-0.263698,0.028718,0.077674,0.057775,0.068756,-0.000270,0.234037,0.074284,0.700467,0.124183,-0.088832,0.088832,-0.144572
workclass,0.040504,1.000000,-0.024338,0.004874,0.003536,-0.020468,0.007110,-0.057947,0.048350,0.031505,0.002644,0.042199,-0.001625,0.002693,0.031261,0.060136,0.015338,-0.071584,0.071584,-0.039233
fnlwgt,-0.076646,-0.024338,1.000000,-0.028145,-0.043195,0.028153,0.000188,0.008931,-0.021291,0.000432,-0.010252,-0.018768,-0.063286,-0.009463,0.000988,-0.069565,-0.004414,-0.026858,0.026858,-0.036700
education,-0.010508,0.004874,-0.028145,1.000000,0.359153,-0.038407,-0.041279,-0.010876,0.014131,0.030046,0.016746,0.055510,0.076060,0.079317,0.029039,0.024049,0.024955,0.027356,-0.027356,0.023466
education_num,0.036527,0.003536,-0.043195,0.359153,1.000000,-0.069304,0.070954,-0.094153,0.031838,0.122630,0.079923,0.148123,0.088894,0.335154,0.117891,0.121482,0.129135,-0.012280,0.012280,-0.058042
marital_status,-0.266288,-0.020468,0.028153,-0.038407,-0.069304,1.000000,0.034962,0.185451,-0.068013,-0.043393,-0.034187,-0.190519,-0.021278,-0.199307,-0.041395,-0.307991,-0.066595,0.129314,-0.129314,0.013292
occupation,0.001739,0.007110,0.000188,-0.041279,0.070954,0.034962,1.000000,-0.037451,-0.004839,0.018021,0.009680,-0.012879,-0.002217,0.034625,0.017437,-0.005742,0.010884,-0.047461,0.047461,-0.031999
relationship,-0.263698,-0.057947,0.008931,-0.010876,-0.094153,0.185451,-0.037451,1.000000,-0.116055,-0.057919,-0.061062,-0.248974,-0.010712,-0.250918,-0.054413,-0.327142,-0.083402,0.582454,-0.582454,0.068117
race,0.028718,0.048350,-0.021291,0.014131,0.031838,-0.068013,-0.004839,-0.116055,1.000000,0.011145,0.018899,0.041910,0.116529,0.071846,0.010081,0.047061,0.024089,-0.087204,0.087204,-0.013113
capital_gain,0.077674,0.031505,0.000432,0.030046,0.122630,-0.043393,0.018021,-0.057919,0.011145,1.000000,-0.031615,0.078409,0.008819,0.223329,0.998521,0.112552,0.564520,-0.048480,0.048480,-0.346996


In [52]:
!pip install ppscore


Defaulting to user installation because normal site-packages is not writeable


In [53]:
import ppscore as pps
df_orig = pd.read_csv('adult_with_headers (1).csv')
df_orig.columns = df_orig.columns.str.strip()
for col in df_orig.select_dtypes('object').columns:
    df_orig[col] = df_orig[col].str.strip()
df_orig.replace('?', np.nan, inplace=True)
df_orig.fillna(df_orig.mode().iloc[0], inplace=True)
print(pps.predictors(df_orig, 'income')[['x','ppscore']]
      .sort_values('ppscore', ascending=False))


ModuleNotFoundError: No module named 'ppscore'